# LLM Interpretability Pipeline - Quick Start

This notebook demonstrates how to use the Interpretable NLP Classification Pipeline
for training a text classifier and understanding its predictions.

In [ ]:
# Install dependencies if needed
# !pip install torch transformers datasets shap lime captum matplotlib seaborn

In [ ]:
import sys
sys.path.append('..')

from src.pipeline import InterpretableNLPPipeline
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

## 1. Load Sample Data

We'll use the IMDB movie review dataset for sentiment classification.

In [ ]:
# Load IMDB dataset
dataset = load_dataset('imdb', split='train[:1000]')  # Using subset for quick demo

# Split into train and validation
train_texts = dataset['text'][:800]
train_labels = dataset['label'][:800]
val_texts = dataset['text'][800:]
val_labels = dataset['label'][800:]

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")
print(f"\nSample review: {train_texts[0][:200]}...")

## 2. Initialize the Pipeline

Create an interpretable pipeline with multiple explanation methods.

In [ ]:
# Initialize pipeline with interpretability methods
pipeline = InterpretableNLPPipeline(
    model_name="distilbert-base-uncased",
    num_labels=2,
    interpretability_methods=["shap", "lime", "attention", "ig"],
    device="auto",  # Will use GPU if available
    max_length=256,
)

print(f"Model: {pipeline.model_name}")
print(f"Device: {pipeline.device}")
print(f"Interpretability methods: {pipeline.interpretability_methods}")

## 3. Train the Model

In [ ]:
# Train the model
history = pipeline.fit(
    train_texts=train_texts,
    train_labels=train_labels,
    val_texts=val_texts,
    val_labels=val_labels,
    epochs=2,
    batch_size=16,
    learning_rate=2e-5,
    label_names=["Negative", "Positive"],
)

print("\nTraining completed!")

In [ ]:
# Plot training history
from src.utils.visualization import plot_training_history

plot_training_history(history['history'])

## 4. Make Predictions

In [ ]:
# Test predictions on some examples
test_texts = [
    "This movie was absolutely fantastic! Great acting and amazing storyline.",
    "Terrible film. Waste of time. The plot made no sense at all.",
    "It was okay, nothing special but not bad either.",
]

predictions, probabilities = pipeline.predict(test_texts, return_probs=True)

for text, pred, prob in zip(test_texts, predictions, probabilities):
    sentiment = "Positive" if pred == 1 else "Negative"
    confidence = prob[pred] * 100
    print(f"Text: {text[:60]}...")
    print(f"Prediction: {sentiment} ({confidence:.1f}% confidence)")
    print()

## 5. Interpretability - Understand the Predictions

### 5.1 SHAP Explanations

In [ ]:
# SHAP explanation for a positive review
positive_text = "This movie was absolutely brilliant! The acting was superb and the story was captivating."

shap_explanation = pipeline.explain_shap(
    text=positive_text,
    num_samples=50,
    visualize=True,
)

print(f"\nPredicted class: {shap_explanation['predicted_class']}")
print(f"Top contributing tokens:")
for attr in shap_explanation['token_attributions'][:5]:
    print(f"  {attr['token']}: {attr['attribution']:.4f}")

### 5.2 LIME Explanations

In [ ]:
# LIME explanation for a negative review
negative_text = "This movie was terrible. Bad acting, boring plot, and a complete waste of money."

lime_explanation = pipeline.explain_lime(
    text=negative_text,
    num_features=10,
    visualize=True,
)

print(f"\nPredicted class: {lime_explanation['predicted_class']}")
print(f"\nTop features:")
for fw in lime_explanation['feature_weights'][:5]:
    direction = "+" if fw['weight'] > 0 else ""
    print(f"  '{fw['word']}': {direction}{fw['weight']:.4f}")

### 5.3 Attention Visualization

In [ ]:
# Visualize attention patterns
attention_text = "The cinematography was stunning but the dialogue felt forced and unnatural."

pipeline.visualize_attention(
    text=attention_text,
    layer=-1,  # Last layer
    aggregate=True,
)

In [ ]:
# Get attention-based token importance
attention_exp = pipeline.explainers['attention'].explain(attention_text)

print("Token importance (from attention):")
sorted_attrs = sorted(
    attention_exp['token_attributions'],
    key=lambda x: x['importance'],
    reverse=True
)[:10]

for attr in sorted_attrs:
    if attr['token'] not in ['[CLS]', '[SEP]', '[PAD]']:
        print(f"  {attr['token']}: {attr['importance']:.4f}")

### 5.4 Integrated Gradients

In [ ]:
# Integrated Gradients explanation
ig_explanation = pipeline.explain_integrated_gradients(
    text="A masterpiece of modern cinema with breathtaking visuals.",
    n_steps=50,
    visualize=True,
)

print(f"\nPredicted class: {ig_explanation['predicted_class']}")

## 6. Batch Explanations

In [ ]:
# Get predictions with all explanations for multiple texts
batch_texts = [
    "One of the best films I've ever seen!",
    "Mediocre at best, forgettable performance.",
]

predictions, explanations = pipeline.predict_with_explanations(
    texts=batch_texts,
    methods=["shap", "attention"],
)

for i, text in enumerate(batch_texts):
    print(f"\nText: {text}")
    print(f"Prediction: {explanations['label_names'][predictions[i]]}")
    print(f"Confidence: {explanations['probabilities'][i][predictions[i]]:.2%}")

## 7. Save and Load the Pipeline

In [ ]:
# Save the trained pipeline
pipeline.save("../outputs/sentiment_model")
print("Pipeline saved!")

In [ ]:
# Load the pipeline
loaded_pipeline = InterpretableNLPPipeline.load("../outputs/sentiment_model")

# Verify it works
pred = loaded_pipeline.predict("Great movie!")
print(f"Loaded model prediction: {loaded_pipeline.label_names[pred[0]]}")

## Summary

In this notebook, we demonstrated:

1. **Training** - Training a DistilBERT classifier on sentiment data
2. **SHAP** - Computing Shapley values for feature importance
3. **LIME** - Generating local interpretable explanations
4. **Attention** - Visualizing transformer attention patterns
5. **Integrated Gradients** - Attribution using gradient integration

Each method provides different insights into model behavior, and using them
together gives a comprehensive understanding of predictions.